In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBRFClassifier

In [18]:
orig_df = pd.read_csv('spam.csv', encoding='Windows-1252')

In [19]:
orig_df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [20]:
orig_df["label"] = (orig_df["v1"] == "spam").astype(int)
orig_df['text'] = orig_df['v2']

drop_cols = ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
orig_df.drop(drop_cols, axis=1, inplace=True)

orig_df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [21]:
X = orig_df["text"]
y = orig_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tf = vectorizer.fit_transform(X_train)
X_test_tf = vectorizer.transform(X_test)

In [22]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "XGBoost": XGBRFClassifier(n_estimators=2000, use_label_encoder=False, eval_metric="logloss", random_state=42, learning_rate=0.01)
}

In [23]:
results = {}

for name, model in models.items():
    model.fit(X_train_tf, y_train)
    y_pred  = model.predict(X_test_tf)
    y_proba = model.predict_proba(X_test_tf)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    results[name] = {"Accuracy": acc, "ROC-AUC": auc, "model": model}

    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  ROC-AUC  : {auc:.4f}")


  Naive Bayes
  Accuracy : 0.9713
  ROC-AUC  : 0.9888

  Logistic Regression
  Accuracy : 0.9704
  ROC-AUC  : 0.9874


c:\Users\Sakshan Sharma\OneDrive\Desktop\codes\machine-learning\nlp_env\Lib\site-packages\xgboost\training.py:199: UserWarning: [22:29:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



  XGBoost
  Accuracy : 0.8664
  ROC-AUC  : 0.9119
